# Starter

In [1]:
# %% [code]
import argparse
from datetime import datetime
import torch
import wandb
import logging
import time
import torch.nn.functional as F

import torch
from data.dataset import Dataset 
import logging
from data.dataset import TestUnit, TestUnits, create_pt_geometric_dataset
from torch_geometric.data import Batch

# Import configuration functions and experiment functions.
from configs.run_model import sw, tcga, dkrl
from configs.utils import str2bool
from experiments.logger import create_logger
from experiments.train import train_and_test, training
from experiments.utils import init_seeds, save_run_results

import numpy as np
import pickle
import os

# Adjust these imports to match your repository's structure.
from data.dataset import Dataset, TestUnit, TestUnits
from units_container import UnitsContainer  # Minimal container defined in a separate file

# Dummy parser that simulates argparse.ArgumentParser for our notebook.
class DummyParser:
    def __init__(self):
        self.args = argparse.Namespace()
    def add_argument(self, name, **kwargs):
        # Remove leading dashes for attribute name.
        key = name.lstrip('-')
        # Set the default value if provided.
        default = kwargs.get("default", None)
        setattr(self.args, key, default)
    def parse_args(self):
        return self.args

def get_args() -> (argparse.Namespace, str, str):
    # Set time strings
    TIME_STR = "{:%Y_%m_%d_%H_%M_%S_%f}".format(datetime.now())
    DATE_STR = "{:%Y_%m_%d}".format(datetime.now())
    
    # Create a dummy parser and add basic arguments.
    parser = DummyParser()
    parser.add_argument("--name", type=str, default=TIME_STR)
    parser.add_argument("--task", type=str, default="dkrl", choices=["dkrl", "sw", "tcga"])
    parser.add_argument("--model", type=str, default="sin", choices=["sin", "gnn", "graphite", "cat", "zero"])
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--cuda", type=int, default=0)
    parser.add_argument("--log_interval", type=int, default=50, help="How many batches to wait before logging training status")
    parser.add_argument("--ablation", type=str, default="False")  # We'll use string and convert later if needed.
    parser.add_argument("--data_path", type=str, default="./generated_data/")
    parser.add_argument("--results_path", type=str, default="./results/")
    
    # Now, based on task, call the appropriate add_params function.
    # These functions expect a parser, so we pass our dummy parser.
    if parser.args.task == "sw":
        sw.add_params(parser)
    elif parser.args.task == "tcga":
        tcga.add_params(parser)
    elif parser.args.task == "dkrl":
        dkrl.add_params(parser)
    
    # Return the resulting Namespace.
    return parser.parse_args(), DATE_STR, TIME_STR



def create_dummy_graphs(treatments):
    """
    Create a dummy id_to_graph_dict for treatments.
    Each treatment gets a dummy graph with one node whose features are the treatment vector.
    """
    n = treatments.shape[0]
    id_to_graph = {}
    for i in range(n):
        id_to_graph[i] = {
            "node_features": treatments[i].reshape(1, -1),  # shape: (1, treatment_dim)
            "edges": np.empty((0, 2)),       # no edges
            "edge_types": np.empty((0,))     # no edge types
        }
    return id_to_graph

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# introduce the DKRL algorithm
import numpy as np
from tqdm import tqdm

def DKRL(y, KZ, KX, N, r, penalty, tol, T, wt = 0, verbose = 0):
    """
    Perform an double kernel representation learning.

    Parameters:
    y (numpy.ndarray): Target vector for optimization.
    KZ (numpy.ndarray): Kernel matrix Z.
    KX (numpy.ndarray): Kernel matrix X.
    N (int): Number of rows in U and V matrices.
    r (int): Number of columns in U and V matrices.
    penalty (float): Regularization parameter.
    tol (float): Tolerance for stopping criterion.
    T (int): Maximum number of iterations.
    wt (float): a number between 0 and 1 for adding identity to degenerate kernel matrix
    

    Returns:
    tuple: Final U and V matrices.
    """
    # Initialize U and V matrices randomly
    Ut = np.random.normal(loc=0, scale=1, size=(N, r))
    Vt = np.random.normal(loc=0, scale=1, size=(N, r))

    for iter in tqdm(range(T)):
        # Update U matrix
        VKX = np.repeat(np.dot(Vt.T, KX), repeats=N, axis=0)
        KZZ = np.tile(KZ, (1, r))
        DesignVt = VKX.T * KZZ  # n * (nr)
        Utt = np.linalg.solve(
            np.dot(DesignVt.T, DesignVt) + penalty * np.kron(np.eye(r, dtype=int), (1-wt) * KZ + wt * np.eye(N)),
            np.dot(DesignVt.T, y)
        )
        Utt = Utt.reshape((N, r), order="F")

        # Update V matrix
        UKZ = np.repeat(np.dot(Utt.T, KZ), repeats=N, axis=0)
        KXX = np.tile(KX, (1, r))
        DesignUt = UKZ.T * KXX  # n * (nr)
        Vtt = np.linalg.solve(
            np.dot(DesignUt.T, DesignUt) + penalty * np.kron(np.eye(r, dtype=int), (1-wt) * KX + wt * np.eye(N)),
            np.dot(DesignUt.T, y)
        )
        Vtt = Vtt.reshape((N, r), order="F")

        # Check stopping criterion
        norm_U = np.linalg.norm(Utt - Ut) / (np.linalg.norm(Ut) + 1e-3)
        norm_V = np.linalg.norm(Vtt - Vt) / (np.linalg.norm(Vt) + 1e-3)
        if verbose:
            print(f"Iteration {iter}: norm_U={norm_U}, norm_V={norm_V}")
        
        
        if norm_U < tol and norm_V < tol:
            break

        # Update Ut and Vt
        Ut = Utt
        Vt = Vtt

    y_pred = np.diag(KZ @ Utt @ Vtt.T @ KX)
    return Utt, Vtt, y_pred

def DKRL_pred(U, V, KZ_pred, KX_pred):
    y_pred = np.diag(KZ_pred.T @ U @ V.T @ KX_pred)
    return y_pred




## SIN 

## Kernel methods

# Monte Carlo

In [3]:
def DGP(r):    
    # import the headlines and embeddings

    df = pd.read_csv("../unique_headlines.csv")  # Replace with your file path
    headlines = np.array(df["headline"].tolist())
    embeddings = np.loadtxt('../embeddings.csv', delimiter=',')

    # Randomly sample a subset of headlines and embeddings for evaluation
    # Set a random seed for reproducibility
    np.random.seed(2025)
    num_actions = 50

    # Randomly sample a subset of headlines and embeddings for evaluation
    # Set a random seed for reproducibility
    ind = np.arange(len(headlines))  # Array with 1000 elements
    sampled_ind = np.random.choice(ind, size=num_actions, replace=False)

    # Randomly sample 100 elements from the array
    headlines = headlines[sampled_ind]
    Z = embeddings[sampled_ind]

    # dimension of the embeddings
    p = len(Z[1])
    print("the dimension of the embeddings is: " + str(len(Z[1])))

    # Simulate covariate level features from Gaussian distributions
    np.random.seed(2025)

    # p = 1000
    # Z = np.random.normal(loc=0, scale=1, size=(N, p))
    # row_norms_Z = np.linalg.norm(Z, axis=1, keepdims=True)
    # Z = Z / row_norms_Z
    # sampled_ind = np.random.choice(range(N), size=N, replace=True)
    # Z = Z[sampled_ind]

    q = 200
    X = np.random.normal(loc=0, scale=1, size=(1000, q))
    row_norms_X = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / row_norms_X

    # Simulate projection matrix
    # generate random projection
    np.random.seed(2025)
    P = np.random.normal(loc=0, scale=1, size=(p, q))
    L, S, Rt = np.linalg.svd(P, full_matrices=False)
    Lr = L[:, :r]
    Rr = Rt[:r, :].T

    return X, Z, num_actions, p, q, Lr, Rr

In [9]:
def run_simulation(replication, args, r, X, Z, num_actions, Lr, Rr):
    # Set seed for this replication
    np.random.seed(2025 + replication)
    torch.manual_seed(2025 + replication)

    # --- Data Generation ---
    # 
    N = 500
    sampled_ind_z = np.random.choice(range(num_actions), size=N, replace=True)
    Zsub = Z[sampled_ind_z]
    sampled_ind_x = np.random.choice(range(num_actions), size=N, replace=True)
    Xsub = X[sampled_ind_x]

    # generate the outcome
    Zrsub = np.dot(Zsub, Lr) # n*r
    Xrsub = np.dot(Xsub, Rr) # n*r
    ysub = np.diag(Zrsub @ Xrsub.T) + 0.001 * np.random.normal(loc = 0, scale = 1, size = N)

    n_samples = Xsub.shape[0]
    n_features = Xsub.shape[1]
    n_treatment_features = Zsub.shape[1]


    # Optionally, you can subsample or re-index (simulate the original code)
    # For simplicity, we use the whole dataset.
    # --- Split Data ---
    split_index = int(0.9 * n_samples)
    X_train, X_test = Xsub[:split_index], Xsub[split_index:]
    Z_train, Z_test = Zsub[:split_index], Zsub[split_index:]
    y_train, y_test = ysub[:split_index], ysub[split_index:]
    
    treatment_ids_train = np.arange(X_train.shape[0])
    treatment_ids_test = np.arange(X_test.shape[0])
    
    # Create dummy edges
    edges_train = np.empty((X_train.shape[0], 0))
    edge_types_train = np.empty((X_train.shape[0], 0))
    edges_test = np.empty((X_test.shape[0], 0))
    edge_types_test = np.empty((X_test.shape[0], 0))
    
    # Create dummy graphs
    id_to_graph_dict_train = create_dummy_graphs(Z_train)
    id_to_graph_dict_test = create_dummy_graphs(Z_test)
    
    # Build SIN training data:
    units_train_dict = {
        "features": X_train,
        "treatments": Z_train,
        "outcomes": y_train,
        "edges": edges_train,
        "edge_types": edge_types_train
    }
    units_train = UnitsContainer(units_train_dict)
    in_sample_dataset_dict = {
        "units": units_train,
        "treatment_ids": treatment_ids_train,
        "id_to_graph_dict": id_to_graph_dict_train,
        "outcomes": y_train
    }
    sin_training_data = Dataset(data_dict=in_sample_dataset_dict)
    
    # For SIN test data, we keep the same format:
    units_test_dict = {
        "features": X_test,
        "treatments": Z_test,
        "outcomes": y_test,
        "edges": edges_test,
        "edge_types": edge_types_test
    }
    units_test = UnitsContainer(units_test_dict)
    out_sample_dataset_dict = {
        "units": units_test,
        "treatment_ids": treatment_ids_test,
        "id_to_graph_dict": id_to_graph_dict_test,
        "outcomes": y_test
    }
    sin_testing_data = Dataset(data_dict=out_sample_dataset_dict)
    
    # --- SIN method ---
    # Train SIN model (replace training() with your actual training function)
    project_name = f"sin_{DATE_STR}-{args.task}" + ("-ABL" if args.ablation else "")
    wandb.init(project=project_name, name=f"{args.model}-{args.seed}", config=args)
    init_seeds(seed=args.seed)

    start_time = time.time()
    model_sin = training(args=args, device=torch.device("cpu"))
    sin_time = time.time() - start_time
    
    # Predict on training data for SIN
    sin_units_train = sin_training_data.data_dict["units"]
    sin_treatment_ids_train = sin_training_data.data_dict["treatment_ids"]
    sin_id_to_graph_train = sin_training_data.data_dict["id_to_graph_dict"]
    pt_train = create_pt_geometric_dataset(
        units=sin_units_train,
        treatment_graphs=[sin_id_to_graph_train[i] for i in sin_treatment_ids_train],
        outcomes=sin_training_data.data_dict["outcomes"]
    )
    with torch.no_grad():
        batch_train = Batch.from_data_list(pt_train)
        pred_train_sin = model_sin.test_prediction(batch_train).cpu().numpy()
    sin_train_error = np.sqrt(np.linalg.norm(y_train - pred_train_sin)**2 / len(y_train))
    
    sin_units_test = sin_testing_data.data_dict["units"]
    sin_treatment_ids_test = sin_testing_data.data_dict["treatment_ids"]
    sin_id_to_graph_test = sin_testing_data.data_dict["id_to_graph_dict"]
    pt_test = create_pt_geometric_dataset(
        units=sin_units_test,
        treatment_graphs=[sin_id_to_graph_test[i] for i in sin_treatment_ids_test],
        outcomes=sin_testing_data.data_dict["outcomes"]
    )
    with torch.no_grad():
        batch_test = Batch.from_data_list(pt_test)
        pred_test_sin = model_sin.test_prediction(batch_test).cpu().numpy()
    sin_test_error = np.sqrt(np.linalg.norm(y_test - pred_test_sin)**2 / len(y_test))
    
    # --- DKRL method ---
    # For kernel methods, we use the full data (or a subsample) and compute kernel matrices.
    # Here we assume that Xsub, Zsub, ysub are the same as X, Z, y (or use our generated ones).
    # Compute linear kernels:
    KZ = np.dot(Zsub, Zsub.T)
    KX = np.dot(Xsub, Xsub.T)
    KZX = KZ * KX

    # Split data into training and testing
    N = n_samples
    N_train = split_index
    N_test = N - split_index
    # Use train/test indices from the splitting above:
    train_id = np.arange(N_train)
    test_id = np.arange(N_train, N)

    # Training kernel matrices
    KZ_train_kernel = KZ[np.ix_(train_id, train_id)]
    KX_train_kernel = KX[np.ix_(train_id, train_id)]
    KZX_train_kernel = KZX[np.ix_(train_id, train_id)]
    y_train_kernel = y_train

    # Testing kernel matrices (use rows corresponding to training set and columns corresponding to test set)
    KZ_test_kernel = KZ[np.ix_(train_id, test_id)]
    KX_test_kernel = KX[np.ix_(train_id, test_id)]
    KZX_test_kernel = KZX[np.ix_(train_id, test_id)]
    y_test_kernel = y_test

    # DKRL parameters:
    penalty = 1e-2
    tol = 5e-2
    T = 200
    
    start_time = time.time()
    U, V, y_hat_dkrl_train = DKRL(y_train_kernel, KZ_train_kernel, KX_train_kernel, N_train, r, penalty, tol, T, wt=0.01)
    dkrl_time = time.time() - start_time


    y_hat_dkrl_test = DKRL_pred(U, V, KZ_test_kernel, KX_test_kernel)
    dkrl_train_error = np.sqrt(np.linalg.norm(y_train_kernel - y_hat_dkrl_train)**2 / N_train)
    dkrl_test_error = np.sqrt(np.linalg.norm(y_test_kernel - y_hat_dkrl_test)**2 / N_test)

    # --- Product Kernel method ---
    penalty_pk = 1.0

    start_time = time.time()
    alpha = np.linalg.solve(KZX_train_kernel + penalty_pk * np.eye(N_train), y_train_kernel)
    pk_time = time.time() - start_time

    y_hat_pk_train = KZX_train_kernel @ alpha
    y_hat_pk_test = KZX_test_kernel.T @ alpha
    pk_train_error = np.sqrt(np.linalg.norm(y_train_kernel - y_hat_pk_train)**2 / N_train)
    pk_test_error = np.sqrt(np.linalg.norm(y_test_kernel - y_hat_pk_test)**2 / N_test)
    
    results = {
        "sin": {"train_error": sin_train_error, "test_error": sin_test_error, "time": sin_time},
        "dkrl": {"train_error": dkrl_train_error, "test_error": dkrl_test_error, "time": dkrl_time},
        "product_kernel": {"train_error": pk_train_error, "test_error": pk_test_error, "time": pk_time},
        "rank": r
    }
    return results

def aggregate_results(results_list):
    agg = {}
    for method in results_list[0].keys():
        if method == "rank":
            continue
        train_errors = np.array([res[method].get("train_error", np.nan) for res in results_list])
        test_errors = np.array([res[method].get("test_error", np.nan) for res in results_list])
        # For methods that report separate training and test times (SIN), take average of both.
        if "train_time" in results_list[0][method]:
            times = np.array([(res[method]["train_time"] + res[method]["test_time"]) / 2 for res in results_list])
        else:
            times = np.array([res[method]["time"] for res in results_list])
        agg[method] = {
            "train_error_mean": np.mean(train_errors),
            "train_error_std": np.std(train_errors),
            "test_error_mean": np.mean(test_errors),
            "test_error_std": np.std(test_errors),
            "time_mean": np.mean(times),
            "time_std": np.std(times),
        }
    return agg

In [14]:
ranks = [2, 3, 5, 7]
num_replications = 100
all_results = {}

args, DATE_STR, TIME_STR = get_args()

for r in ranks:
    print(f"Running Monte Carlo simulation for rank: {r}")
    results_sample = []
    X, Z, num_actions, p, q, Lr, Rr = DGP(r)
    for rep in range(num_replications):
        print(f"  Replication {rep+1}/{num_replications}")
        res = run_simulation(rep, args, r, X, Z, num_actions, Lr, Rr)
        results_sample.append(res)
    all_results[r] = results_sample

np.save("test-over-rank-exp.npy", all_results)    

Running Monte Carlo simulation for rank: 2
the dimension of the embeddings is: 384
  Replication 1/100


como_val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▃▃▃▄▄▅▅▅▅▅▆▆▆▇▇▇████▁▂▂▂▂▃▄▄▄▅▅▆▆▆▆
train_como_loss,▁█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_global_model_loss,▄▂▂▅▂▄▂▄▅█▂▃█▃▄▄▄▄▄▃▇▂▂▃▆▁▇▃▆▆▇▁▅▇▇▅
train_propensity_loss,▇█▇▇▆▅▅▆▅▅▅▄▃▄▃▃▃▂▂▂▃▁▂▂▃▂▂▁▂▂▂▁▂▁▂▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁
como_val_loss,0.00019
epoch,36
train_como_loss,8e-05
train_global_model_loss,0.0001
train_propensity_loss,0.10638


  2%|▎         | 5/200 [00:00<00:06, 32.18it/s]

  Replication 2/100


como_val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▂▃▃▃▃▄▄▄▄▄▆▆▆▇▇███▂▂▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆
train_como_loss,▁█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_global_model_loss,▄▂▂▅▂▄▂▄▅█▂▃█▃▄▄▄▄▄▃▇▂▂▃▆▁▇▃▆▆▇▁▅▇▇▅
train_propensity_loss,▇█▇▇▆▅▅▆▅▅▅▄▃▄▃▃▃▂▂▂▃▁▂▂▃▂▂▁▂▂▂▁▂▁▂▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁
como_val_loss,0.00019
epoch,36
train_como_loss,8e-05
train_global_model_loss,0.0001
train_propensity_loss,0.10638


...

como_val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▂▂▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇██▁▁▁▃▃▃▄▅▅▅▅▆
train_como_loss,▁█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_global_model_loss,▄▂▂▅▂▄▂▄▅█▂▃█▃▄▄▄▄▄▃▇▂▂▃▆▁▇▃▆▆▇▁▅▇▇▅
train_propensity_loss,▇█▇▇▆▅▅▆▅▅▅▄▃▄▃▃▃▂▂▂▃▁▂▂▃▂▂▁▂▂▂▁▂▁▂▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁
como_val_loss,0.00019
epoch,36
train_como_loss,8e-05
train_global_model_loss,0.0001
train_propensity_loss,0.10638


  4%|▍         | 8/200 [00:03<01:20,  2.39it/s]

  Replication 100/100


como_val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇██▁▁▂▂▂▃▃▄▄▄▄▅▆▆
train_como_loss,▁█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_global_model_loss,▄▂▂▅▂▄▂▄▅█▂▃█▃▄▄▄▄▄▃▇▂▂▃▆▁▇▃▆▆▇▁▅▇▇▅
train_propensity_loss,▇█▇▇▆▅▅▆▅▅▅▄▃▄▃▃▃▂▂▂▃▁▂▂▃▂▂▁▂▂▂▁▂▁▂▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁
como_val_loss,0.00019
epoch,36
train_como_loss,8e-05
train_global_model_loss,0.0001
train_propensity_loss,0.10638


  4%|▍         | 9/200 [00:03<01:18,  2.43it/s]


In [16]:
agg_results = {r: aggregate_results(all_results[r]) for r in ranks}
print("Monte Carlo Simulation Results:")
for r, methods in agg_results.items():
    print(f"Rank: {r}")
    for method, metrics in methods.items():
        print(f"  Method: {method}")
        for key, value in metrics.items():
            print(f"    {key}: {value}")
            
# Save aggregated results to file
# output_errors_path = "./generated_data/dkrl/seed-0/bias-0.1/monte_carlo_results.npz"
# np.savez(output_errors_path, **agg_results)
# print("Saved Monte Carlo results to", output_errors_path)

Monte Carlo Simulation Results:
Rank: 2
  Method: sin
    train_error_mean: 0.01047816011490104
    train_error_std: 0.0006121898250330425
    test_error_mean: 0.01035315918671402
    test_error_std: 0.0016658649969443404
    time_mean: 8.46858118534088
    time_std: 0.6583983413435994
  Method: dkrl
    train_error_mean: 0.0019326955846080266
    train_error_std: 0.0003909752469278121
    test_error_mean: 0.0035850122916268566
    test_error_std: 0.0010227038241828691
    time_mean: 0.19066327571868896
    time_std: 0.03393831069496701
  Method: product_kernel
    train_error_mean: 0.004343582548707471
    train_error_std: 0.00020801453359872897
    test_error_mean: 0.008013103277349219
    test_error_std: 0.0013015459502382248
    time_mean: 0.0014123892784118652
    time_std: 0.00023988347986051128
Rank: 3
  Method: sin
    train_error_mean: 0.011307220268474598
    train_error_std: 0.0005840077455597735
    test_error_mean: 0.011333851140382071
    test_error_std: 0.001613237633924

In [ ]:
# Define the metrics for each rank.
metrics = ["Train Error Mean", "Train Error Std", "Test Error Mean", "Test Error Std", "Time Mean", "Time Std"]

# Extract the sorted ranks
ranks = sorted(agg_results.keys())

# Create a MultiIndex for the columns: first level = rank, second level = metric
columns = pd.MultiIndex.from_product([ranks, metrics], names=["Rank", "Metric"])

# Assume that all ranks have the same set of methods.
methods = list(agg_results[ranks[0]].keys())

# Build a dictionary where each key is a method and its value is a list of metric values.
data = {}
for method in methods:
    row_values = []
    for r in ranks:
        metrics_data = agg_results[r][method]
        row_values.append(metrics_data["train_error_mean"])
        row_values.append(metrics_data["train_error_std"])
        row_values.append(metrics_data["test_error_mean"])
        row_values.append(metrics_data["test_error_std"])
        row_values.append(metrics_data["time_mean"])
        row_values.append(metrics_data["time_std"])
    data[method] = row_values

# Create a DataFrame with methods as rows and our MultiIndex columns.
df = pd.DataFrame.from_dict(data, orient="index", columns=columns)

df.to_csv('upworthy-output.csv', index=False)




Rank                          2                                           \
Metric         Train Error Mean Test Error Mean Test Error Std Time Mean   
sin                    0.010478        0.010353       0.001666  8.468581   
dkrl                   0.001933        0.003585       0.001023  0.190663   
product_kernel         0.004344        0.008013       0.001302  0.001412   

Rank                                    3                                 \
Metric          Time Std Train Error Mean Test Error Mean Test Error Std   
sin             0.658398         0.011307        0.011334       0.001613   
dkrl            0.033938         0.002206        0.004848       0.001269   
product_kernel  0.000240         0.004801        0.009005       0.001278   

Rank                                              5                  \
Metric         Time Mean  Time Std Train Error Mean Test Error Mean   
sin             7.810295  0.707330         0.014309        0.014439   
dkrl            0.506221 